# HumanEval Benchmark — 4 Models

This notebook runs the first **25 HumanEval questions** against:

1. `deepseek-ai/DeepSeek-R1`
2. `meta-llama/Llama-3.1-8B-Instruct`
3. `o3-mini`
4. `gpt-4o-mini`

It records the generated answer and per-question generation time in one CSV file: `humaneval_4_models_results.csv`.

**Important:** The full `DeepSeek-R1` model is extremely large and is unlikely to fit into ordinary free Colab GPU memory. The notebook catches model-loading errors and records them instead of stopping the benchmark. If you need R1 locally in Colab, use an appropriate quantized/inference-provider setup.

## 1. Install dependencies

In [1]:
!pip -q install -U transformers accelerate datasets huggingface_hub openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.4 MB/s eta 0:00:00


## 2. Enter API tokens securely

Do **not** paste tokens directly into the notebook code. Colab will hide the input while typing.

If you previously exposed your Hugging Face token publicly, revoke/rotate it first.

In [ ]:
import os
from getpass import getpass

HF_TOKEN = ""
OPENAI_API_KEY = ""

if not HF_TOKEN:
    raise ValueError("Hugging Face token is required.")

if not OPENAI_API_KEY:
    raise ValueError("OpenAI API key is required.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("API credentials loaded securely for this Colab session.")

API credentials loaded securely for this Colab session.


## 3. Imports and configuration

In [3]:
import os
import gc
import csv
import time
import pandas as pd
import torch

from datasets import load_dataset
from huggingface_hub import login
from transformers import pipeline
from openai import OpenAI

NUM_QUESTIONS = 25
OUTPUT_CSV = "humaneval_4_models_results.csv"

HF_MODELS = [
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    #"meta-llama/Llama-3.1-8B-Instruct",
]

OPENAI_MODELS = [
    "o3-mini",
    "gpt-4o-mini",
]

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 4. Login and load the first 25 HumanEval questions

In [4]:
login(token=os.environ["HF_TOKEN"])
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("Loading HumanEval dataset...")
dataset = load_dataset("openai/openai_humaneval", split="test")
questions = dataset.select(range(NUM_QUESTIONS))

print(f"Total HumanEval questions available: {len(dataset)}")
print(f"Questions selected for benchmark: {len(questions)}")
print("First task:", questions[0]["task_id"])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading HumanEval dataset...


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Total HumanEval questions available: 164
Questions selected for benchmark: 25
First task: HumanEval/0


## 5. Benchmark helper functions

The timer starts immediately before the model request/generation and ends immediately after the response is received. Model loading time is not included in the per-question generation time.

In [5]:
results = []

SYSTEM_PROMPT = (
    "You are an expert Python programmer. "
    "Solve the HumanEval programming problem. "
    "Return only the completed Python code. "
    "Do not explain your answer."
)

def add_result(item, question_number, model_name, answer, elapsed, status="success", error=""):
    results.append({
        "task_id": item["task_id"],
        "question_number": question_number,
        "model": model_name,
        "prompt": item["prompt"],
        "answer": answer,
        "generation_time_seconds": round(elapsed, 4),
        "status": status,
        "error": error,
    })

def cleanup_model(pipe):
    try:
        del pipe
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


## 6. Run Hugging Face models

Only one Hugging Face model is loaded at a time to reduce GPU memory usage.

In [6]:
def run_huggingface_model(model_name):
    print("\n" + "=" * 80)
    print("HUGGING FACE MODEL:", model_name)
    print("=" * 80)

    pipe = None

    try:
        load_start = time.perf_counter()
        pipe = pipeline(
            "text-generation",
            model=model_name,
            device_map="auto",
            torch_dtype="auto",
            trust_remote_code=True,
        )
        load_time = time.perf_counter() - load_start
        print(f"Model loaded in {load_time:.2f} seconds")
    except Exception as e:
        error_text = f"MODEL_LOAD_ERROR: {type(e).__name__}: {e}"
        print(error_text)
        for index, item in enumerate(questions, start=1):
            add_result(item, index, model_name, "", 0, "error", error_text)
        cleanup_model(pipe)
        return

    for index, item in enumerate(questions, start=1):
        print(f"Question {index}/{NUM_QUESTIONS} - {item['task_id']}")

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["prompt"]},
        ]

        start = time.perf_counter()
        try:
            response = pipe(
                messages,
                max_new_tokens=512,
                do_sample=False,
                return_full_text=False,
            )
            elapsed = time.perf_counter() - start
            generated = response[0]["generated_text"]

            if isinstance(generated, list):
                # Some chat pipelines return a message list.
                generated = generated[-1].get("content", str(generated))

            add_result(item, index, model_name, str(generated), elapsed)
            print(f"  Time: {elapsed:.4f}s")
        except Exception as e:
            elapsed = time.perf_counter() - start
            error_text = f"{type(e).__name__}: {e}"
            add_result(item, index, model_name, "", elapsed, "error", error_text)
            print("  ERROR:", error_text)

    cleanup_model(pipe)

for model_name in HF_MODELS:
    run_huggingface_model(model_name)



HUGGING FACE MODEL: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

  Time: 340.5881s
Question 23/25 - HumanEval/22


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Time: 340.1155s
Question 24/25 - HumanEval/23


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Time: 213.6927s
Question 25/25 - HumanEval/24
  Time: 339.4248s


## 7. Run OpenAI models

In [ ]:
def run_openai_model(model_name):
    print("\n" + "=" * 80)
    print("OPENAI MODEL:", model_name)
    print("=" * 80)

    for index, item in enumerate(questions, start=1):
        print(f"Question {index}/{NUM_QUESTIONS} - {item['task_id']}")

        start = time.perf_counter()
        try:
            response = openai_client.responses.create(
                model=model_name,
                instructions=SYSTEM_PROMPT,
                input=item["prompt"],
                max_output_tokens=1000,
            )

            elapsed = time.perf_counter() - start
            answer = response.output_text

            add_result(item, index, model_name, answer, elapsed)
            print(f"  Time: {elapsed:.4f}s")
        except Exception as e:
            elapsed = time.perf_counter() - start
            error_text = f"{type(e).__name__}: {e}"
            add_result(item, index, model_name, "", elapsed, "error", error_text)
            print("  ERROR:", error_text)

for model_name in OPENAI_MODELS:
    run_openai_model(model_name)



OPENAI MODEL: o3-mini
Question 1/25 - HumanEval/0
  Time: 3.9075s
Question 2/25 - HumanEval/1
  Time: 7.8199s
Question 3/25 - HumanEval/2
  Time: 3.2688s
Question 4/25 - HumanEval/3
  Time: 1.9204s
Question 5/25 - HumanEval/4
  Time: 3.1863s
Question 6/25 - HumanEval/5
  Time: 3.6242s
Question 7/25 - HumanEval/6
  Time: 4.3716s
Question 8/25 - HumanEval/7
  Time: 1.6507s
Question 9/25 - HumanEval/8
  Time: 1.8963s
Question 10/25 - HumanEval/9
  Time: 2.0303s
Question 11/25 - HumanEval/10
  Time: 5.7419s
Question 12/25 - HumanEval/11
  Time: 3.5959s
Question 13/25 - HumanEval/12
  Time: 2.2272s
Question 14/25 - HumanEval/13
  Time: 4.5046s
Question 15/25 - HumanEval/14
  Time: 4.2101s
Question 16/25 - HumanEval/15
  Time: 2.7520s
Question 17/25 - HumanEval/16
  Time: 3.8123s
Question 18/25 - HumanEval/17
  Time: 3.2501s
Question 19/25 - HumanEval/18
  Time: 4.6347s
Question 20/25 - HumanEval/19
  Time: 3.0709s
Question 21/25 - HumanEval/20
  Time: 4.5677s
Question 22/25 - HumanEval/21


## 8. Save all 100 results to one CSV

In [7]:
columns = [
    "task_id",
    "question_number",
    "model",
    "prompt",
    "answer",
    "generation_time_seconds",
    "status",
    "error",
]

df = pd.DataFrame(results, columns=columns)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("BENCHMARK COMPLETE")
print("=" * 80)
print("Rows:", len(df))
print("Expected rows:", NUM_QUESTIONS * 4)
print("CSV:", OUTPUT_CSV)
print("\nRows per model:")
print(df.groupby("model").size())

display(df[["task_id", "question_number", "model", "generation_time_seconds", "status"]].head(20))


BENCHMARK COMPLETE
Rows: 25
Expected rows: 100
CSV: humaneval_4_models_results.csv

Rows per model:
model
deepseek-ai/DeepSeek-R1-Distill-Qwen-7B    25
dtype: int64


,task_id,question_number,model,generation_time_seconds,status
0,HumanEval/0,1,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,355.1028,success
1,HumanEval/1,2,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,341.4894,success
2,HumanEval/2,3,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,340.0339,success
3,HumanEval/3,4,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,340.5831,success
4,HumanEval/4,5,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,340.1335,success
5,HumanEval/5,6,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,339.5289,success
6,HumanEval/6,7,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,340.2065,success
7,HumanEval/7,8,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,339.5422,success
8,HumanEval/8,9,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,339.7984,success
9,HumanEval/9,10,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,339.5545,success


## 9. Download the CSV

In [8]:
from google.colab import files
files.download(OUTPUT_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Basic timing summary

This summarizes the recorded generation time for successful runs. It does not evaluate whether the generated code passes the HumanEval tests.

In [ ]:
successful = df[df["status"] == "success"].copy()

if len(successful) > 0:
    summary = (
        successful.groupby("model")["generation_time_seconds"]
        .agg(["count", "mean", "median", "min", "max", "sum"])
        .reset_index()
    )
    display(summary)
else:
    print("No successful runs to summarize.")